# Project #21 - Zero-to-Hero Notebook\n\nProduction-Grade Text Classification Framework with Transformers, PEFT/LoRA, Optuna, ONNX, MLflow, FastAPI and Streamlit.

## 1) Environment and Config\nSwitch between quick and full modes using `CONFIG_PATH`.

In [ ]:
from textclf_framework.settings import load_config\nCONFIG_PATH = 'configs/quick.yaml'  # change to configs/default.yaml for full run\ncfg = load_config(CONFIG_PATH)\ncfg

## 2) Data Pipeline\nLoad datasets, apply preprocessing, profile class/length statistics, and save reproducibility manifest.

In [ ]:
from textclf_framework.data.loader import DatasetLoader\nfrom textclf_framework.data.preprocessing import PreprocessConfig\nfrom textclf_framework.data.profiling import build_dataset_profile\nfrom textclf_framework.data.versioning import build_manifest\n\nloader = DatasetLoader(seed=cfg.seed, val_size=cfg.datasets.validation_size)\nds = loader.load(cfg.datasets.primary, preprocess_config=PreprocessConfig())\nprofile = build_dataset_profile(cfg.datasets.primary, ds)\nmanifest = build_manifest(cfg.datasets.primary, 'huggingface', ds, cfg.seed, {}, loader.label_names(ds))\nprofile, manifest

## 3) Fine-Tuning\nRun Hugging Face Trainer with full fine-tuning or LoRA strategy.

In [ ]:
from textclf_framework.training.engine import TextClassificationTrainer, TrainingRunConfig\n\nlabels = loader.label_names(ds)\ntrain_cfg = TrainingRunConfig(\n    output_dir=cfg.paths.artifact_dir / 'notebook_run',\n    model_name='distilbert',\n    strategy='full',\n    learning_rate=cfg.training.learning_rate,\n    weight_decay=cfg.training.weight_decay,\n    epochs=cfg.training.epochs,\n    train_batch_size=cfg.training.train_batch_size,\n    eval_batch_size=cfg.training.eval_batch_size,\n    gradient_accumulation_steps=cfg.training.gradient_accumulation_steps,\n    warmup_ratio=cfg.training.warmup_ratio,\n    max_length=cfg.training.max_length,\n    gradient_checkpointing=cfg.training.gradient_checkpointing,\n    fp16=cfg.training.fp16,\n)\ntrainer = TextClassificationTrainer(train_cfg)\n# result = trainer.train(ds, num_labels=len(labels), label_names=labels)

## 4) Explainability\nLIME, SHAP and attention modules are available in `textclf_framework.explainability`.

In [ ]:
from textclf_framework.explainability.lime_shap import TextExplainer\nexplainer = TextExplainer(class_names=labels)\n# lime_result = explainer.explain_with_lime('Example text', predict_proba_fn)

## 5) ONNX Optimization\nExport and benchmark with ONNX Runtime.

In [ ]:
from textclf_framework.optimization.onnx_utils import ONNXExporter\n# onnx_path = ONNXExporter.export(model, tokenizer, 'artifacts/onnx/model.onnx')

## 6) Serving\n- FastAPI app: `uv run uvicorn apps.fastapi_app:app --reload`\n- Streamlit UI: `uv run streamlit run apps/streamlit_app.py`